# SF Case Files — The Profile
> *"Every suspect has a history. I profile them all. Thoroughly."* — Adrian Monk
This notebook runs the **Vertipaq Analyzer** against the SFCaseFiles semantic model using Semantic Link Labs.
Vertipaq Analyzer reveals how your model is physically stored in memory:
table and column sizes, cardinality, dictionary sizes, segment counts, and relationship stats.
This is how you find the suspects that are quietly bloating your model.
---
| Section | Theme | What it does |
|---|---|---|
| The Initial Scan | First look at the scene | Run Vertipaq Analyzer, display results inline |
| The Usual Suspects | Who's taking up space? | Pandas analysis — largest tables, high-cardinality columns |
| Filing the Evidence | Preserve everything | Export to .zip for DAX Studio / Tabular Editor |
| Building the Case File | Persistent records | Export to delta tables for trending over time |
| Cold Case Review | Reviewing old files | Import and visualize a previously saved .zip |


---
## Setup

In [2]:
%run 00_Setup_Fresh

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 4, Finished, Available, Finished, True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
✅ semantic-link-labs 0.14.0 installed
✅ Workspace    : FabConSQLCon2026
✅ Workspace ID : 21df5422-28b3-46aa-a1c5-ae2cd2a48578
✅ Lakehouse    : EvidenceLocker
✅ Lakehouse ID : a5f11fba-90dd-4722-81a9-d9cc0ebc6fdc
✅ Setup complete — the investigation begins


In [3]:
DATASET   = "Trudoso10K"
DATASETXL = "Trudoso10K_WithAutoDateXL"
WORKSPACE = "FabConSQLCon2026"

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 5, Finished, Available, Finished, False)

---
## Before We Profile — The Auto Date/Time Tax
> *"I can tell you exactly what's wrong. I just need to show you the number."*
**Trudoso10K** — our sample dataset, named for Trudy Monk. Same data as the Microsoft Contoso sample,
but a better name for a detective session.
These two models are identical except for one setting: **Auto Date/Time**.
One has it on. One doesn't. Let's see what that costs.

In [4]:
# Baseline model — Auto Date/Time OFF
print("=== Trudoso10K (Auto Date/Time OFF) ===")
vpa_base = labs.vertipaq_analyzer(dataset=DATASET, workspace=WORKSPACE)

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 6, Finished, Available, Finished, False)

=== Trudoso10K (Auto Date/Time OFF) ===


In [5]:
# Same model — Auto Date/Time ON
print("=== Trudoso10K_WithAutoDateXL (Auto Date/Time ON) ===")
vpa_xl = labs.vertipaq_analyzer(dataset=DATASETXL, workspace=WORKSPACE)

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 7, Finished, Available, Finished, False)

=== Trudoso10K_WithAutoDateXL (Auto Date/Time ON) ===


---
## What You Just Saw
Every date column with Auto Date/Time enabled generates hidden date tables behind the scenes —
one per date column. Multiply that across a real-world model and size compounds fast.
**10x–100x** is not an exaggeration for large models with many date columns.
One setting. `vertipaq_analyzer` catches it in seconds.

---
## The Initial Scan
> *"I notice things other people don't notice. It's a gift. And a curse."*
Run Vertipaq Analyzer against SFCaseFiles and display all results inline.
The output is a dictionary of DataFrames — one per object type:
- **Tables** — row count, total size, % of model
- **Columns** — cardinality, encoding, dictionary size, data size
- **Relationships** — cardinality, missing rows, max from/to values
- **Partitions** — segment count, records per segment
- **Hierarchies** — levels, user hierarchy size

In [6]:
# Run Vertipaq Analyzer — displays results inline
vpa = labs.vertipaq_analyzer(dataset=DATASET, workspace=WORKSPACE)

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 8, Finished, Available, Finished, False)

In [7]:
# Inspect available result categories
print("Result tables available:")
for name in vpa.keys():
    print(f"  - {name}")

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 9, Finished, Available, Finished, False)

Result tables available:
  - Model Summary
  - Tables
  - Partitions
  - Columns
  - Relationships
  - Hierarchies


In [8]:
# Display each result table
for name, df in vpa.items():
    print(f"\n=== {name} ===")
    display(df)

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 10, Finished, Available, Finished, False)


=== Model Summary ===


SynapseWidget(Synapse.DataFrame, 57bbf270-0eec-4900-89cd-844a7aee2917)


=== Tables ===


SynapseWidget(Synapse.DataFrame, f2ab9d5d-d63a-42a4-824b-56b38937f9b0)

...

SynapseWidget(Synapse.DataFrame, a066a5ff-7614-4a8f-b786-e55e206bdb3d)


=== Relationships ===


SynapseWidget(Synapse.DataFrame, f891627b-5810-4707-b765-18773be3a666)


=== Hierarchies ===


SynapseWidget(Synapse.DataFrame, 4d832ced-f685-4453-a956-ab7deec74c2c)

---
## The Usual Suspects — Pandas Analysis
> *"There are always patterns. Always. You just have to know where to look."*
Pull the Vertipaq results into pandas and interrogate them directly.
Three questions to answer: which tables are largest, which columns have the highest cardinality,
and are any columns VALUE-encoded when they could be HASH?

In [9]:
df_tables  = vpa.get("Tables")
df_columns = vpa.get("Columns")

# Table memory footprint — ranked largest to smallest
if df_tables is not None:
    size_col = [c for c in df_tables.columns if "size" in c.lower() or "byte" in c.lower()]
    print("Table columns:", df_tables.columns.tolist())
    display(df_tables.sort_values(df_tables.columns[2], ascending=False))

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 11, Finished, Available, Finished, False)

Table columns: ['Table Name', 'Type', 'Row Count', 'Total Size', 'Dictionary Size', 'Data Size', 'Hierarchy Size', 'Relationship Size', 'User Hierarchy Size', 'Partitions', 'Columns', '% DB']


SynapseWidget(Synapse.DataFrame, 58fe425f-73d1-442e-a420-c5f2fe2c83d7)

In [10]:
# Column cardinality — who has the most distinct values?
# High cardinality + VALUE encoding = memory pressure
if df_columns is not None:
    print("Column columns:", df_columns.columns.tolist())
    
    # Find cardinality column
    card_col = [c for c in df_columns.columns if "cardinality" in c.lower()]
    if card_col:
        display(
            df_columns[[
                c for c in df_columns.columns 
                if any(x in c.lower() for x in ["table", "column", "cardinality", "encoding", "size"])
            ]].sort_values(card_col[0], ascending=False).head(15)
        )

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 12, Finished, Available, Finished, False)

Column columns: ['Table Name', 'Column Name', 'Type', 'Cardinality', 'Total Size', 'Data Size', 'Dictionary Size', 'Hierarchy Size', '% Table', '% DB', 'Data Type', 'Encoding', 'Is Resident', 'Temperature', 'Last Accessed']


SynapseWidget(Synapse.DataFrame, ebc1a464-9f0e-44f1-aeed-c456079333ba)

In [11]:
# Spotlight: VALUE-encoded columns (less efficient than HASH)
# These are the suspects worth investigating for data type optimization
if df_columns is not None:
    enc_col = [c for c in df_columns.columns if "encoding" in c.lower()]
    if enc_col:
        df_value_encoded = df_columns[df_columns[enc_col[0]] == "VALUE"]
        print(f"VALUE-encoded columns: {len(df_value_encoded)}")
        display(df_value_encoded)

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 13, Finished, Available, Finished, False)

VALUE-encoded columns: 0


SynapseWidget(Synapse.DataFrame, 5bfc1ee5-a981-44f2-945e-74abb5b1a0e8)

In [12]:
# Relationships — check for high cardinality joins
df_rels = vpa.get("Relationships")
if df_rels is not None:
    print("Relationship columns:", df_rels.columns.tolist())
    display(df_rels)

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 14, Finished, Available, Finished, False)

Relationship columns: ['From Object', 'To Object', 'Multiplicity', 'Used Size', 'Max From Cardinality', 'Max To Cardinality', 'Missing Rows']


SynapseWidget(Synapse.DataFrame, 6e212770-1ec7-4ef9-bfba-bae17387ef82)

---
## Filing the Evidence — Export to .zip
> *"Always make a copy. Evidence gets lost. I've seen it."*
Export results to a **.zip file** saved to your lakehouse `Files/` folder.
The .zip is compatible with **DAX Studio** and **Tabular Editor** — import it for
deeper offline analysis or share it with a colleague who doesn't have model access.

In [13]:
# Export to .zip — saved to Files/ in attached lakehouse
labs.vertipaq_analyzer(dataset=DATASET, workspace=WORKSPACE, export='zip')

print("✅ Vertipaq Analyzer results exported to .zip in your lakehouse Files/ folder.")
print("   Open in DAX Studio or Tabular Editor for offline analysis.")

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 15, Finished, Available, Finished, False)

🟢 The Vertipaq Analyzer info for the 'Trudoso10K' semantic model in the 'FabConSQLCon2026' workspace has been saved to the 'Vertipaq Analyzer/FabConSQLCon2026.Trudoso10K.zip' in the default lakehouse attached to this notebook.
✅ Vertipaq Analyzer results exported to .zip in your lakehouse Files/ folder.
   Open in DAX Studio or Tabular Editor for offline analysis.


---
## Building the Case File — Export to Delta Tables
> *"One observation tells you nothing. Patterns over time tell you everything."*
Export results to **delta tables** in your lakehouse. Each run appends a new snapshot —
so you can trend model size and cardinality across releases, sprints, or deployments.
Connect Power BI directly to these tables for a model health history dashboard.
This is the governance play: schedule it, watch the trends, catch regressions before users do.

In [14]:
# Export to delta tables — appends on each run
labs.vertipaq_analyzer(dataset=DATASET, workspace=WORKSPACE, export='table')

print("✅ Vertipaq Analyzer results appended to delta tables in your lakehouse.")
print("   Run on a schedule to build historical model size trends.")

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 16, Finished, Available, Finished, False)

⌛ Saving Vertipaq Analyzer to delta tables in the lakehouse...

🟢 The dataframe has been saved as the 'vertipaqanalyzer_columns' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.
🟢 The dataframe has been saved as the 'vertipaqanalyzer_tables' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.
🟢 The dataframe has been saved as the 'vertipaqanalyzer_partitions' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.
🟢 The dataframe has been saved as the 'vertipaqanalyzer_relationships' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.
🟢 The dataframe has been saved as the 'vertipaqanalyzer_hierarchies' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.
🟢 The dataframe has been saved as the 'vertipaqanalyzer_model' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.
✅ Vertipaq Analyzer results appended to delta tabl

---
## Cold Case Review — Import from .zip
> *"Sometimes you have to go back. The answer was always in the old files."*
Already have a Vertipaq .zip from a previous export or from DAX Studio?
`import_vertipaq_analyzer()` loads and visualizes it directly in the notebook —
no live model connection required. Useful for reviewing historical snapshots,
comparing before/after a change, or analyzing a model you can't connect to directly.
Update `folder_path` and `file_name` before uncommenting.

In [15]:
# Visualize a previously exported .zip file
# folder_path: location in your lakehouse Files/ folder
# file_name: the .zip filename (without path)

# labs.import_vertipaq_analyzer(
#     folder_path='/lakehouse/default/Files/',
#     file_name='SFCaseFiles_vpa.zip'
# )

print("Cold Case Review cell ready — update folder_path and file_name, then uncomment to run.")

StatementMeta(, 85d23b55-97a1-4eb5-99e2-f83c7ac76acb, 17, Finished, Available, Finished, False)

Cold Case Review cell ready — update folder_path and file_name, then uncomment to run.


---

## Summary

| What | Function | Export |
|---|---|---|
| Run and display inline | `labs.vertipaq_analyzer(dataset, workspace)` | None |
| Export to .zip for DAX Studio / TE | `labs.vertipaq_analyzer(..., export='zip')` | `.zip` in Files/ |
| Export to delta tables for trending | `labs.vertipaq_analyzer(..., export='table')` | Appends to tables |
| Visualize saved .zip | `labs.import_vertipaq_analyzer(folder_path, file_name)` | — |

### What to look for

| Signal | Why it matters |
|---|---|  
| High cardinality columns | More distinct values = larger dictionary = more memory |
| VALUE encoding | Less efficient than HASH; often fixable with data type change |
| Tables with unexpectedly high row counts | Might indicate a fanout or modeling issue |
| Large dictionary relative to data size | Suggests high-cardinality text column — consider integer surrogate key |
| Relationships with missing rows | Referential integrity gaps — potential filter context surprises |

> *"Now I know everything about this model. Everything."* — Adrian Monk

---

### Resources
- [Semantic Link Labs GitHub](https://github.com/microsoft/semantic-link-labs)
- [Model Optimization helper notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Model%20Optimization.ipynb)
- [Vertipaq Analyzer video](https://www.youtube.com/watch?v=RnrwUqg2-VI)
- [DAX Studio — free Vertipaq Analyzer UI](https://daxstudio.org)
- [thedaxshepherd.com](https://thedaxshepherd.com)
